# مرحله ۲: منحنی سرعت در برابر کیفیت

سؤال: وقتی تعداد گام‌های دیکدر (NFE) را کم می‌کنیم، چقدر سرعت می‌گیریم و چقدر کیفیت از دست می‌دهیم؟

روش: همان ۱۰ جمله‌ی ثابت را با NFE برابر ۱، ۲، ۴، ۱۰ و ۲۵ تولید می‌کنیم. برای هر NFE، زمان کل، مدت کل صدا و RTF را حساب می‌کنیم، یک جمله را در گام‌های مختلف گوش می‌دهیم و بعد با Whisper کوچک، WER را می‌سنجیم. نقطه‌ی زانویی جایی است که با بیشتر کردن گام، بهبود کیفیت خیلی کم می‌شود ولی زمان خیلی بیشتر می‌شود.

نکته‌ی مهم درباره‌ی RTF: در این نوت‌بوک، RTF دیواری است؛ یعنی زمان کل اجرا (شامل بارگذاری مدل و وکودر) تقسیم بر مدت کل صدا، سرشکن‌شده روی ۱۰ جمله. این عدد با RTF مقاله‌ی DriftTTS که وکودر و بارگذاری را کنار می‌گذارد، مستقیم قابل مقایسه نیست. روند تغییرات در برابر NFE همان چیزی است که برای پروژه مهم است، نه عدد مطلق.
این نوت‌بوک خودکفا است: چون هر نوت‌بوک Colab ماشین مخصوص خودش را دارد، محیط پایتون ۳.۱۱ و Matcha-TTS باید داخل همین نوت‌بوک ساخته شوند. پس از بالا به پایین و یکی‌یکی اجرا کن؛ قدم‌های پیش‌نیاز را رد نکن.


## پیش‌نیاز: ساختن محیط ۳.۱۱ داخل همین نوت‌بوک

این همان کاری است که در مرحله‌ی ۱-ب انجام شد، ولی چون نوت‌بوک جدید ماشین تازه‌ای دارد، باید دوباره در همین‌جا انجام شود. این بخش چند دقیقه طول می‌کشد؛ صبر کن تا هر سلول تیک سبز بخورد.


In [ ]:
!apt-get update -qq
!apt-get install -y espeak-ng
!pip install -q uv
!uv --version


In [ ]:
!uv python install 3.11
!uv venv --python 3.11 /content/matcha311
!/content/matcha311/bin/python --version


In [ ]:
!uv pip install --python /content/matcha311/bin/python --upgrade pip
!uv pip install --python /content/matcha311/bin/python torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!uv pip install --python /content/matcha311/bin/python matcha-tts


In [ ]:
!/content/matcha311/bin/python -c "import torch; print('torch:', torch.__version__); print('CUDA:', torch.cuda.is_available())"
!/content/matcha311/bin/matcha-tts --help | head -n 5


## قدم ۰: بررسی GPU و محیط ۳.۱۱

اگر Runtime عوض شده یا قطع شده باشد، این قدم نشان می‌دهد محیط matcha311 هنوز هست یا نه. اگر فایل matcha-tts پیدا نشد، اول به نوت‌بوک مرحله‌ی ۱-ب برگرد.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!ls /content/matcha311/bin/matcha-tts
!grep -n 'weights_only' /content/matcha311/lib/python3.11/site-packages/matcha/cli.py || echo 'PATCH MISSING - run the patch cell below'

## قدم ۰-ب: وصله‌ی بارگذاری چک‌پوینت (در صورت نیاز)

این دستور فقط در صورتی تغییری می‌دهد که وصله‌ی weights_only=False هنوز اعمال نشده باشد؛ اگر قبلا اعمال شده، تغییری نمی‌کند. چک‌پوینت رسمی Matcha منبع مطمئنی است، پس این گزینه فقط برای همین فایل رسمی به کار می‌رود.

In [ ]:
!sed -i "s/MatchaTTS.load_from_checkpoint(checkpoint_path, map_location=device)/MatchaTTS.load_from_checkpoint(checkpoint_path, map_location=device, weights_only=False)/" /content/matcha311/lib/python3.11/site-packages/matcha/cli.py
!grep -n 'weights_only' /content/matcha311/lib/python3.11/site-packages/matcha/cli.py

## قدم ۱: ۱۰ جمله‌ی ثابت

این ۱۰ جمله در همه‌ی NFE ها ثابت می‌مانند. جمله‌ها ساده، بدون عدد و بدون مخفف هستند تا تبدیل متن به واج تمیز باشد. طول جمله‌ها هم متنوع است، ولی همه در یک بازه‌ی معمولی‌اند.

In [ ]:
sentences = [
    'The quick brown fox jumps over the lazy dog.',
    'She opened the window and looked at the garden.',
    'The train arrived at the station just after sunset.',
    'He read the book slowly, one page at a time.',
    'The children played in the park until it started to rain.',
    'A small bird sang on the branch above the quiet river.',
    'The old house stood at the end of a long, dusty road.',
    'She wrote a letter to her friend about the summer holiday.',
    'The baker made fresh bread every morning before sunrise.',
    'They walked home together under a sky full of stars.',
]
open('sentences.txt', 'w', encoding='utf-8').write('\n'.join(sentences) + '\n')
print('sentences written:', len(sentences))
for i, s in enumerate(sentences, 1):
    print(i, s)

## قدم ۲: سنتز برای همه‌ی NFE ها

برای هر NFE، یک بار مدل بارگذاری می‌شود و ۱۰ جمله به‌صورت دسته‌ای تولید می‌شوند. به همین دلیل زمان هر NFE شامل یک بار بارگذاری مدل است، ولی چون روی ۱۰ جمله سرشکن می‌شود، روند مقایسه سالم می‌ماند. این قدم چند دقیقه طول می‌کشد. اگر در یک NFE خطا گرفتی یا تعداد فایل‌های wav برابر ۱۰ نشد، در همان NFE توقف کن و خطا را بفرست.

In [ ]:
import json
import pathlib
import subprocess
import time

nfes = [1, 2, 4, 10, 25]
timing = {}
for nfe in nfes:
    out = pathlib.Path(f'stage2_outputs/nfe_{nfe}')
    out.mkdir(parents=True, exist_ok=True)
    print(f'\n=== NFE={nfe} ===', flush=True)
    t0 = time.time()
    r = subprocess.run(['/content/matcha311/bin/matcha-tts', '--file', 'sentences.txt', '--batched', '--steps', str(nfe), '--output_folder', str(out)], capture_output=True, text=True)
    wall = time.time() - t0
    timing[nfe] = wall
    print(f'returncode={r.returncode} wall={wall:.1f}s')
    print('stdout tail:', r.stdout[-800:])
    print('stderr tail:', r.stderr[-800:])
    wavs = sorted(out.glob('utterance_*.wav'))
    print(f'wav files: {len(wavs)}')
    if r.returncode != 0 or len(wavs) != 10:
        print('STOP: check the error above before continuing.')
        break
pathlib.Path('timing.json').write_text(json.dumps(timing, indent=2))
print(timing)

## قدم ۳: محاسبه‌ی RTF دیواری

برای هر NFE، مدت همه‌ی فایل‌های wav را جمع می‌کنیم و زمان دیواری را بر آن تقسیم می‌کنیم. RTF زیر ۱ یعنی سریع‌تر از زمان واقعی، ولی یادت باشد این RTF شامل بارگذاری مدل و وکودر هم هست.

In [ ]:
import contextlib
import csv
import json
import pathlib
import wave

timing = json.loads(pathlib.Path('timing.json').read_text())
rows = []
for nfe_str, wall in timing.items():
    nfe = int(nfe_str)
    out = pathlib.Path(f'stage2_outputs/nfe_{nfe}')
    total = 0.0
    count = 0
    for wav in sorted(out.glob('utterance_*.wav')):
        with contextlib.closing(wave.open(str(wav))) as w:
            total += w.getnframes() / w.getframerate()
            count += 1
    rtf = wall / total if total else None
    row = {'nfe': nfe, 'sentences': count, 'total_duration_s': round(total, 2), 'wall_time_s': round(wall, 1), 'rtf_wall': round(rtf, 4) if rtf else None}
    rows.append(row)
    print(row)
with open('rtf_results.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

## قدم ۴: گوش دادن؛ جمله‌ی ۳ در گام‌های مختلف

جمله‌ی ۳ این است: The train arrived at the station just after sunset. همین جمله را با NFE برابر ۱، ۱۰ و ۲۵ گوش بده و بنویس در کدام‌یک تلفظ خراب‌تر، صدا مبهم‌تر یا مکث‌ها غیرطبیعی‌تر است. این بخش، همان تست شنیداری کوچک پروژه است.

In [ ]:
from IPython.display import Audio, display

for nfe in [1, 10, 25]:
    path = f'stage2_outputs/nfe_{nfe}/utterance_003.wav'
    print(f'NFE={nfe}: {path}')
    display(Audio(path))

## قدم ۵: نصب ابزار رونویسی برای WER

برای WER، از نسخه‌ی سریع Whisper با مدل tiny استفاده می‌کنیم. مدل tiny دقیق‌ترین Whisper نیست، ولی چون برای همه‌ی NFE ها همان مدل به کار می‌رود، مقایسه‌ی نسبی سالم می‌ماند. عدد مطلق WER با مقاله که از مدل بزرگ استفاده کرده، مستقیم قابل مقایسه نیست.

In [ ]:
!apt-get update -qq
!apt-get install -y ffmpeg
!uv pip install --python /content/matcha311/bin/python faster-whisper jiwer

## قدم ۶: نوشتن اسکریپت رونویسی

این سلول فقط یک فایل به نام transcribe.py می‌سازد؛ هنوز رونویسی نمی‌کند. ترتیب فایل‌ها utterance_001 تا utterance_010 است و با ترتیب جمله‌ها در sentences.txt یکی است.

In [ ]:
lines = [
    'import csv, pathlib',
    'from faster_whisper import WhisperModel',
    'from jiwer import wer',
    '',
    'sentences = [line.strip() for line in open("sentences.txt", encoding="utf-8") if line.strip()]',
    'model = WhisperModel("tiny", device="cpu", compute_type="int8")',
    'rows = []',
    'base = pathlib.Path("stage2_outputs")',
    'for nfe_dir in sorted(base.glob("nfe_*")):',
    '    nfe = int(nfe_dir.name.split("_")[1])',
    '    for i, wav in enumerate(sorted(nfe_dir.glob("utterance_*.wav"))):',
    '        if i >= len(sentences):',
    '            continue',
    '        segments, info = model.transcribe(str(wav), language="en")',
    '        hyp = " ".join(seg.text.strip() for seg in segments).strip()',
    '        ref = sentences[i]',
    '        w = wer(ref, hyp)',
    '        rows.append({"nfe": nfe, "sentence_id": i + 1, "reference": ref, "hypothesis": hyp, "wer": w, "wav": str(wav)})',
    '        print(f"nfe={nfe} sentence={i + 1} wer={w:.3f} hyp={hyp}", flush=True)',
    'with open("wer_results.csv", "w", newline="", encoding="utf-8") as f:',
    '    writer = csv.DictWriter(f, fieldnames=["nfe", "sentence_id", "reference", "hypothesis", "wer", "wav"])',
    '    writer.writeheader()',
    '    writer.writerows(rows)',
    'print(f"wrote wer_results.csv rows={len(rows)}")',
]
open('transcribe.py', 'w', encoding='utf-8').write('\n'.join(lines) + '\n')
print('transcribe.py written')

## قدم ۷: اجرای رونویسی

این قدم چند دقیقه طول می‌کشد؛ در اولین اجرا مدل tiny دانلود می‌شود. اگر خطا گرفتی، در همین قدم توقف کن.

In [ ]:
!/content/matcha311/bin/python transcribe.py

## قدم ۸: جدول نهایی و نمودارها

میانگین WER هر NFE را با RTF همان NFE کنار هم می‌گذاریم. نمودار اول RTF در برابر NFE، نمودار دوم WER در برابر NFE، و نمودار سوم همان منحنی بده‌بستان است: RTF در برابر WER. نقطه‌ی زانویی را با چشم روی نمودار سوم پیدا کن؛ جایی که با بیشتر کردن NFE، WER خیلی کم نمی‌شود ولی RTF خیلی بیشتر می‌شود.

In [ ]:
import csv
import os
import pathlib

import matplotlib.pyplot as plt

rtf = {}
with open('rtf_results.csv', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        rtf[int(row['nfe'])] = row
summary = []
if os.path.exists('wer_results.csv'):
    wers = {}
    with open('wer_results.csv', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            wers.setdefault(int(row['nfe']), []).append(float(row['wer']))
    for nfe in sorted(rtf):
        mean_wer = sum(wers.get(nfe, [])) / len(wers[nfe]) if wers.get(nfe) else None
        summary.append({'nfe': nfe, 'rtf_wall': float(rtf[nfe]['rtf_wall']), 'mean_wer': round(mean_wer, 4) if mean_wer is not None else None, 'wall_time_s': float(rtf[nfe]['wall_time_s']), 'total_duration_s': float(rtf[nfe]['total_duration_s'])})
else:
    print('wer_results.csv not found; plotting RTF only.')
    for nfe in sorted(rtf):
        summary.append({'nfe': nfe, 'rtf_wall': float(rtf[nfe]['rtf_wall']), 'mean_wer': None, 'wall_time_s': float(rtf[nfe]['wall_time_s']), 'total_duration_s': float(rtf[nfe]['total_duration_s'])})
for row in summary:
    print(row)
with open('summary.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=summary[0].keys())
    writer.writeheader()
    writer.writerows(summary)
nfes = [row['nfe'] for row in summary]
rtfs = [row['rtf_wall'] for row in summary]
plt.figure()
plt.plot(nfes, rtfs, marker='o')
plt.xlabel('NFE')
plt.ylabel('RTF (wall)')
plt.title('RTF vs NFE')
plt.savefig('plot_rtf_vs_nfe.png', dpi=150)
plt.show()
if summary[0]['mean_wer'] is not None:
    wers = [row['mean_wer'] for row in summary]
    plt.figure()
    plt.plot(nfes, wers, marker='o')
    plt.xlabel('NFE')
    plt.ylabel('Mean WER')
    plt.title('WER vs NFE')
    plt.savefig('plot_wer_vs_nfe.png', dpi=150)
    plt.show()
    plt.figure()
    plt.plot(rtfs, wers, marker='o')
    for x, y, nfe in zip(rtfs, wers, nfes):
        plt.annotate(str(nfe), (x, y))
    plt.xlabel('RTF (wall)')
    plt.ylabel('Mean WER')
    plt.title('Speed-quality tradeoff (labels = NFE)')
    plt.savefig('plot_tradeoff.png', dpi=150)
    plt.show()

## چک‌لیست موفقیت مرحله ۲

- [ ] برای هر NFE، ۱۰ فایل wav ساخته شد.
- [ ] جدول RTF چاپ شد و با بیشتر شدن NFE، زمان هم بیشتر شد.
- [ ] جمله‌ی ۳ را در NFE های ۱، ۱۰ و ۲۵ گوش دادی و تفاوت را یادداشت کردی.
- [ ] جدول WER چاپ شد و summary.csv ساخته شد.
- [ ] سه نمودار دیده شد و نقطه‌ی زانویی را مشخص کردی.

## محدودیت‌هایی که در گزارش بنویس

- RTF این نوت‌بوک دیواری است و شامل بارگذاری مدل و وکودر می‌شود؛ با RTF مقاله که وکودر را کنار می‌گذارد، مستقیم مقایسه نمی‌شود.
- دمای نمونه‌برداری روی پیش‌فرض ۰.۶۶۷ ثابت مانده، پس تکرار همان جمله ممکن است کمی فرق کند.
- WER با مدل tiny سنجیده شده، پس عدد مطلقش از مقاله که مدل بزرگ به کار برده، بالاتر است؛ مقایسه‌ی نسبی بین NFE ها معتبر است.
- فقط ۱۰ جمله و یک گوینده‌ی انگلیسی است؛ پس نتیجه را به فارسی یا به گوینده‌های دیگر تعمیم نده.

## چه چیزی را برایم بفرستی

فایل summary.csv، سه نمودار، و یک جمله درباره‌ی اینکه به نظر تو نقطه‌ی زانویی کدام NFE است و در گوش دادن، NFE برابر ۱ چه تفاوتی با ۱۰ داشت. اگر در قدمی خطا گرفتی، همان قدم و ۳۰ خط آخرش را بفرست.